In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def summarize_dataset(images, masks, name=""):
    print(f"--- {name} ---")
    print(f"Imágenes: {images.shape}, dtype={images.dtype}, min={images.min():.3f}, max={images.max():.3f}")
    print(f"Máscaras: {masks.shape}, dtype={masks.dtype}, min={masks.min():.3f}, max={masks.max():.3f}")
    n_classes = masks.shape[-1] if masks.ndim == 4 else 1
    print(f"Número de clases en máscara: {n_classes}")

In [ ]:
def plot_class_balance(masks, class_names=None):
    """
    Proporción de píxeles por clase
    """
    n_classes = masks.shape[-1]
    if class_names is None:
        class_names = [f"clase_{i}" for i in range(n_classes)]

    pixel_counts = masks.sum(axis=(0, 1, 2))
    proportions = pixel_counts / pixel_counts.sum()

    plt.figure(figsize=(6, 4))
    plt.bar(class_names, proportions)
    plt.ylabel("Proporción de píxeles")
    plt.title("Balance de clases en las máscaras")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    for name, prop in zip(class_names, proportions):
        print(f"{name}: {prop:.4%}")
        
    return proportions

In [ ]:
def count_empty_masks(masks, background_idx=0):
    """
    Cuenta cuántas imágenes no tienen lesión (100% fondo).
    """
    n_classes = masks.shape[-1]
    lesion_channels = [i for i in range(n_classes) if i != background_idx]
    has_lesion = masks[..., lesion_channels].sum(axis=(1, 2, 3)) > 0

    n_empty = (~has_lesion).sum()
    n_total = len(masks)
    print(f"Imágenes sin lesión: {n_empty}/{n_total} ({n_empty/n_total:.1%})")
    return has_lesion

In [ ]:
def visualize(image_batch, mask_batch=None, pred_batch=None, num_samples=8, hot_encode=True):
    """Muestra imagen + cada canal de máscara (y predicción opcional) lado a lado."""
    num_classes = mask_batch.shape[-1] if mask_batch is not None else 0
    fig, ax = plt.subplots(num_classes + 1, num_samples, figsize=(num_samples * 2, (num_classes + 1) * 2))

    for i in range(num_samples):
        ax_image = ax[0, i] if num_classes > 0 else ax[i]
        if hot_encode:
            ax_image.imshow(image_batch[i, :, :, 0], cmap='Greys')
        else:
            ax_image.imshow(image_batch[i, :, :])
        ax_image.set_xticks([])
        ax_image.set_yticks([])

        if mask_batch is not None:
            for j in range(num_classes):
                if pred_batch is None:
                    mask_to_show = mask_batch[i, :, :, j]
                else:
                    mask_to_show = np.zeros(shape=(*mask_batch.shape[1:-1], 3))
                    mask_to_show[..., 0] = pred_batch[i, :, :, j] > 0.5
                    mask_to_show[..., 1] = mask_batch[i, :, :, j]
                ax[j + 1, i].imshow(mask_to_show, vmin=0, vmax=1)
                ax[j + 1, i].set_xticks([])
                ax[j + 1, i].set_yticks([])

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_hists(images1, images2=None, label1="dataset 1", label2="dataset 2"):
    """
    Compara distribución de intensidades de píxel entre dos datasets
    """
    plt.hist(images1.ravel(), bins=100, density=True, color='b', alpha=1 if images2 is None else 0.5, label=label1)
    if images2 is not None:
        plt.hist(images2.ravel(), bins=100, density=True, alpha=0.5, color='orange', label=label2)
        plt.legend()
    plt.title("Distribución de intensidades" if images2 is None else "Comparación de distribuciones")
    plt.xlabel("Valor de píxel")
    plt.ylabel("Densidad")
    plt.show()

In [ ]:
def run_eda(images, masks, name="train", class_names=None):
    summarize_dataset(images, masks, name=name)

    has_lesion = count_empty_masks(masks)
    proportions = plot_class_balance(masks, class_names=class_names)
    visualize(images[:8], masks[:8])

    return {
        "n_samples": len(images),
        "has_lesion": has_lesion,
        "class_proportions": proportions,
    }